# 一般參數（模型自己學的）：  
  Linear Regression 的斜率、截距  
  → 從資料裡自動學出來  

# 超參數（你自己設定的）：  
  max_depth, alpha, learning_rate  
  → 模型不會自己學  
  → 要你手動設定  
  → 設不好 → 過擬合或欠擬合  
# GridSearchCV      → 暴力搜尋所有組合
# RandomizedSearchCV → 隨機搜尋部分組合
# Optuna            → 智慧搜尋（進階）




In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split,GridSearchCV, RandomizedSearchCV

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
base_model = RandomForestClassifier(random_state=42)
base_model.fit(X_train, y_train)
print(f"基準準確率: {accuracy_score(y_test, base_model.predict(X_test)):.4f}")

基準準確率: 0.9000


#  GridSearchCV 

#  K-Fold Cross Validation（K 折交叉驗證）
把資料分成 K 份，輪流當「測試集」，其餘當「訓練集」  
第1輪  
Train: 2,3,4,5  
Test : 1  
第2輪  
Train: 1,3,4,5  
Test : 2  

...一直到第5輪  

# 暴力搜尋 參數組合 

叫模型幫我試不同設定，找最好的Random Forest

n_estimators:    [100, 200, 300]      → 3種   Random Forest 有幾棵樹 
max_depth:       [3, 5, 7]            → 3種   限制樹長多深
min_samples_split: [2, 5, 10]         → 3種   一個節點至少要多少資料才能再切
總組合 = 3 × 3 × 3 = 27種  
每種跑 5 折 K-Fold  
總共跑 27 × 5 = 135 次  
取平均最高的參數組合  

#  cv=5, #5-fold cross validation
#  scoring='accuracy',#評估標準是： 準確率 = 對的 / 全部
# n_jobs = 用幾個 CPU 核心來跑
n_jobs=1  → 用1個核心，慢
n_jobs=2  → 用2個核心
n_jobs=-1 → 用全部核心，最快


In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5, 
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"最佳參數: {grid_search.best_params_}")
print(f"最佳準確率: {grid_search.best_score_:.4f}")
print(f"測試集準確率: {accuracy_score(y_test, grid_search.predict(X_test)):.4f}")

最佳參數: {'max_depth': 7, 'min_samples_split': 10, 'n_estimators': 200}
最佳準確率: 0.8900
測試集準確率: 0.8850


# RandomizedSearchCV
GridSearchCV：  
  搜尋所有組合  
  27種全部跑完  
  保證找到設定範圍內最好的  

RandomizedSearchCV：  
  從範圍裡隨機抽取組合  
  n_iter=20 → 只跑20種組合  
  不保證找到最好的  
  但速度快很多  
# from scipy.stats import randint
SciPy 的「隨機變數分佈」版本
randint(100, 500) 100~499隨機選1.個數
# n_iter=20
範圍內隨機抽20 組參數組合

In [9]:
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 15),
    'min_samples_split': randint(2, 20)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print(f"最佳參數: {random_search.best_params_}")
print(f"最佳準確率: {random_search.best_score_:.4f}")
print(f"測試集準確率: {accuracy_score(y_test, random_search.predict(X_test)):.4f}")

最佳參數: {'max_depth': 9, 'min_samples_split': 10, 'n_estimators': 266}
最佳準確率: 0.8950
測試集準確率: 0.8950


# Optuna 貝葉斯式超參數優化
參數少 → GridSearchCV  
參數多 → RandomizedSearchCV  
追求最佳 → Optuna（下一個） 

# optuna.logging.set_verbosity(optuna.logging.WARNING)
不要印太多 Optuna 訊息  
只顯示 warning 以上  



#  def objective(trial):  //objective = 目標函數  

Optuna 每次試一組參數  
把這組參數丟進 objective  
objective 回傳一個分數  
Optuna 根據這個分數決定下次要試哪組參數  

不是隨機試（RandomizedSearch）  
而是根據之前的結果智慧決定下一組  
越試越聰明  

# trial.suggest_int('max_depth', 3, 15)
在 3~15 中選一個整數,但Optuna 會根據前面試的結果決定下一次更可能試哪裡

# **params
params是一個 dictionary（字典） 
**叫做 dictionary unpacking（字典解包）
# RandomForestClassifier(**params, random_state=42)
等價於：  
RandomForestClassifier(  
    n_estimators=300,  
    max_depth=7,  
    min_samples_split=5,  
    random_state=42  
)  



# return accuracy_score(y_test, model.predict(X_test))
告訴 Optuna：「這組參數的好壞」  

# study = optuna.create_study(direction='maximize')
# direction
direction='maximize' → 分數越高越好（準確率）  
direction='minimize' → 分數越低越好（用在 MSE 這類誤差指標）

# study.optimize(objective, n_trials=50)
objective 是def objective(trial):的 objective  
n_trials=50 訓練50次  

In [16]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20)
    }
    
    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"最佳參數: {study.best_params}")
print(f"最佳準確率: {study.best_value:.4f}")
print(f"最佳模型: {study.best_trial}")

最佳參數: {'n_estimators': 393, 'max_depth': 13, 'min_samples_split': 2}
最佳準確率: 0.9050
最佳模型: FrozenTrial(number=5, state=<TrialState.COMPLETE: 1>, values=[0.905], datetime_start=datetime.datetime(2026, 6, 3, 2, 22, 46, 791163), datetime_complete=datetime.datetime(2026, 6, 3, 2, 22, 47, 446985), params={'n_estimators': 393, 'max_depth': 13, 'min_samples_split': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=100, step=1), 'max_depth': IntDistribution(high=15, log=False, low=3, step=1), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1)}, trial_id=5, value=None)


基準:             0.9000
GridSearch:       0.8850
RandomizedSearch: 0.8950
Optuna:           0.9050  ← 最好

GridSearch    → 暴力搜尋固定組合
RandomizedSearch → 隨機搜尋
Optuna        → 智慧搜尋
               根據之前結果決定下一組
               50次試驗比27次更有效率

# 超參數調整整理
GridSearchCV：  
  → 參數少時用  
  → 保證找到最好的組合  
  → 組合多時很慢  

RandomizedSearchCV：  
  → 參數多時用  
  → 快，但不保證最佳  
  → n_iter 控制搜尋次數  

Optuna：  
  → 競賽推薦  
  → 智慧搜尋，越試越準  
  → 可以設定時間限制  
  → n_trials 控制搜尋次數  

# 那為何暴力搜尋固定組合找不到Optuna 找到的最佳參數?
GridSearch 的參數範圍：  
  n_estimators: [100, 200, 300]  
  max_depth: [3, 5, 7]  
  min_samples_split: [2, 5, 10]  

Optuna 找到的最佳參數：  
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20)
  n_estimators: 359  ← 不在 GridSearch 的選項裡  
  max_depth: 13      ← 不在 GridSearch 的選項裡  
  min_samples_split: 6 ← 不在 GridSearch 的選項裡  

GridSearch 的品質取決於你設定的選項  
選項設得好 → 結果好  
選項設得差 → 找不到最佳值  
 
Optuna 範圍更靈活  
適合不知道最佳值在哪裡的時候  